In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm


import sys
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre


import torch.nn.functional as F
from pathlib import Path


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import time
import pickle
import networkx as nx
from probeinterface import write_probeinterface, read_probeinterface, Probe



In [2]:
def count_array2_in_range_of_array1(array1, array2, threshold=5):

    sorted_array1 = np.sort(array1)
    array2 = np.sort(array2)
    
    lefts = array2 - threshold
    rights = array2 + threshold
    
    left_indices = np.searchsorted(sorted_array1, lefts, side='left')
    
    right_indices = np.searchsorted(sorted_array1, rights, side='right')
    
    has_within_range = right_indices > left_indices
    
    count = np.sum(has_within_range)
    
    return count

def label_array1_based_on_array2(array1, array2, threshold=5):
    array_1 = np.sort(array1)
    sorted_array2 = np.sort(array2)
    
    labels = np.zeros(len(array1), dtype=int)
    
    for i, value in enumerate(array1):
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def detect_local_maxima_in_window(data, window_size=20, std_multiplier=2):

    """
    在每个滑动窗口范围内检测局部最大值的索引，并确保最大值大于两倍的标准差。

    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_rows, n_columns)。
    window_size : int
        滑动窗口的大小，用于定义局部范围，默认为 20。
    std_multiplier : float
        标准差的倍数，用于筛选局部最大值，默认为 2。

    返回:
    local_maxima_indices : list of numpy.ndarray
        每行局部最大值的索引列表，每个元素是对应行局部最大值的索引数组。
    """
    local_maxima_indices = []

    for row in data:
        maxima_indices = []
        row_std = np.std(row.astype(np.float32))
        threshold = std_multiplier * row_std

        for start in range(0, len(row), window_size):
            end = min(start + window_size, len(row))
            window = np.abs(row[start:end])
            
            if len(window) > 0:
                local_max_index = np.argmax(window)
                local_max_value = window[local_max_index]
                
                if local_max_value > threshold:
                    maxima_indices.append(start + local_max_index)  
        
        local_maxima_indices.extend(maxima_indices)
        local_maxima_indices = list(set(local_maxima_indices))  

    return local_maxima_indices


def cluster_label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的 'time' 和 'cluster' 对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 的 'time' 中，则标记为对应的 'cluster' 值，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        包含 'time' 和 'cluster' 的二维数组。
        第一列为 'time'，第二列为 'cluster'。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 array2 中的 'cluster' 或 0。
    """

    array2 = np.array(array2.iloc[:, [5, 1]])
    sorted_indices = np.argsort(array2[:, 0])
    sorted_array2 = array2[sorted_indices]
    
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2[:, 0], left, side='left')
        right_index = np.searchsorted(sorted_array2[:, 0], right, side='right')
        
        # 如果范围内存在值，则标记为对应的 'cluster'
        if right_index > left_index:
            # 获取范围内的第一个匹配值的 'cluster'
            labels[i] = sorted_array2[left_index, 1]
    
    return labels


def label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的值对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 中，则标记为 1，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        用于判断的数组。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 0 或 1。
    """
    # 对 array2 进行排序以加速搜索
    sorted_array2 = np.sort(array2)
    
    # 初始化标签数组，默认值为 0
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        # 使用二分搜索判断范围内是否存在值
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        # 如果范围内存在值，则标记为 1
        if right_index > left_index:
            labels[i] = 1
    
    return labels


def extract_windows(data, indices, window_size=61):
    """
    根据给定的时间点索引提取窗口。
    
    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_channels, time)
    indices : numpy.ndarray
        时间点索引数组，用于指定需要提取窗口的中心点
    window_size : int
        窗口长度，默认为61（对应time-30到time+31）
    
    返回:
    windows : numpy.ndarray
        提取的窗口数据，形状为 (len(indices), n_channels, window_size)
    """
    n_channels, time_length = data.shape
    half_window = window_size // 2

    if np.any(indices < half_window) or np.any(indices >= time_length - half_window):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        window = data[:, idx - half_window:idx + half_window + 1]
        windows.append(window)

    windows = np.array(windows)
    return windows

In [3]:
from scipy.io import loadmat


probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])

probe_loc = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
probe.set_device_channel_indices(probe_loc['probeloc'].values)

In [4]:
recording_raw = se.read_intan(f"/home/ubuntu/Downloads/grid/M190011_250521_141514_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)
recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

In [5]:
spike_inf = pd.read_csv("/media/ubuntu/sda/duan/script/spike_sorting/spike_inf.tsv", index_col=0, sep='\t')
cluster_inf = pd.read_csv("/media/ubuntu/sda/duan/script/spike_sorting/cluster_inf.csv", index_col=0)

In [6]:
def create_best_channels_group_to_clusters_dict(cluster_inf_df):
    """
    根据best_channels列创建best_channels组合到clusters的映射字典
    
    Args:
        cluster_inf_df: 包含best_channels列的cluster信息DataFrame
    
    Returns:
        dict: {best_channels_tuple: [cluster_id1, cluster_id2, ...]}
    """
    best_channels_group_to_clusters = {}
    
    for idx, row in cluster_inf_df.iterrows():
        cluster_id = row['cluster_id']
        best_channels_str = row['best_channels']
        
        if pd.notna(best_channels_str) and best_channels_str != 'None':
            try:
                # 解析字符串格式的通道列表，例如 "[1, 2, 3]"
                best_channels = eval(best_channels_str)  # 将字符串转换为列表
                
                # 将通道列表转换为元组作为字典的key（因为列表不能作为字典key）
                best_channels_tuple = tuple(sorted(best_channels))
                
                if best_channels_tuple not in best_channels_group_to_clusters:
                    best_channels_group_to_clusters[best_channels_tuple] = []
                best_channels_group_to_clusters[best_channels_tuple].append(cluster_id)
                    
            except (ValueError, SyntaxError) as e:
                print(f"警告: 无法解析cluster {cluster_id}的best_channels: {best_channels_str}")
                continue
    
    # 对每个通道组合的cluster列表进行排序
    for channels_tuple in best_channels_group_to_clusters:
        best_channels_group_to_clusters[channels_tuple].sort()
    
    return best_channels_group_to_clusters

best_channels_group_dict = create_best_channels_group_to_clusters_dict(cluster_inf)

In [8]:
# Spike Classification 阶段
print("=== 开始Spike Classification阶段 ===")

# 1. 筛选出cluster数量大于1的best_channels_group_dict条目
multi_cluster_groups = {}
for channels_tuple, cluster_ids in best_channels_group_dict.items():
    if len(cluster_ids) > 1:
        multi_cluster_groups[channels_tuple] = cluster_ids

print(f"找到 {len(multi_cluster_groups)} 个包含多个cluster的通道组合:")
for channels_tuple, cluster_ids in multi_cluster_groups.items():
    print(f"  通道组合 {list(channels_tuple)}: clusters {cluster_ids}")

if len(multi_cluster_groups) == 0:
    print("警告: 没有找到包含多个cluster的通道组合，无法进行classification训练")
else:
    print(f"\n将对这些 {len(multi_cluster_groups)} 个通道组合进行spike classification训练")


=== 开始Spike Classification阶段 ===
找到 17 个包含多个cluster的通道组合:
  通道组合 [199, 208, 210, 240, 241, 242]: clusters [2, 27, 40, 76, 78]
  通道组合 [93, 159, 172, 175, 250, 251]: clusters [5, 11]
  通道组合 [127, 152, 204, 219, 253, 254]: clusters [13, 14]
  通道组合 [190, 201, 205, 237, 239, 255]: clusters [16, 59]
  通道组合 [12, 13, 77, 90, 92, 94]: clusters [26, 130]
  通道组合 [29, 30, 46, 138, 141, 173]: clusters [51, 52]
  通道组合 [46, 142, 157, 158, 173, 206]: clusters [53, 54, 74]
  通道组合 [186, 202, 217, 222, 248, 249]: clusters [61, 62]
  通道组合 [216, 217, 223, 233, 234, 248]: clusters [64, 66, 67]
  通道组合 [183, 216, 218, 232, 233, 246]: clusters [70, 71, 72]
  通道组合 [177, 208, 210, 241, 242, 245]: clusters [79, 91]
  通道组合 [17, 34, 98, 113, 129, 146]: clusters [97, 98, 102]
  通道组合 [34, 51, 97, 98, 113, 129]: clusters [107, 109, 110]
  通道组合 [5, 6, 37, 52, 103, 119]: clusters [120, 121]
  通道组合 [2, 4, 5, 52, 71, 87]: clusters [123, 124, 125]
  通道组合 [0, 7, 19, 32, 117, 182]: clusters [129, 131]
  通道组合 [147, 176, 192, 

In [9]:
# 2. 创建Spike Classification模型类
class Spike_Classification_MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, n_channels, time_window):
        super(Spike_Classification_MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(hidden_size2, 32)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        self.fc4 = nn.Linear(32, output_size)
        
        self.n_channels = n_channels
        self.time_window = time_window
        
    def forward(self, x):
        x = x.reshape(-1, self.n_channels * self.time_window)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

# 3. 数据准备函数 - 直接从spike_inf提取时间点
def prepare_classification_data_from_spike_inf(recording_f, spike_inf_temp, cluster_ids, 
                                            valid_channels, window_size=91):
    """
    直接从spike_inf中提取时间点，准备classification数据
    
    参数:
    recording_f: 记录数据
    spike_inf_temp: 对应clusters的spike信息DataFrame
    cluster_ids: cluster ID列表
    valid_channels: 有效通道列表
    window_size: 时间窗大小
    
    返回:
    windows: 提取的时间窗数据
    labels: 对应的cluster标签
    """
    half_window = window_size // 2
    windows = []
    labels = []
    
    print(f"开始从spike_inf提取数据，总共 {len(spike_inf_temp)} 个spike...")
    
    # 按时间排序
    spike_inf_sorted = spike_inf_temp.sort_values('time')
    
    for idx, row in tqdm(spike_inf_sorted.iterrows(), total=len(spike_inf_sorted), desc="提取时间窗"):
        spike_time = int(row['time'])
        cluster_id = int(row['cluster_id'])
        
        # 检查时间点是否在有效范围内
        if spike_time >= half_window and spike_time < recording_f.get_num_samples() - half_window:
            try:
                # 提取时间窗数据
                window_data = recording_f.get_traces(
                    start_frame=spike_time - half_window,
                    end_frame=spike_time + half_window + 1,
                    channel_ids=valid_channels
                )  # shape: (n_channels, window_size)
                
                # 转换为 (window_size, n_channels) 格式
                window_data = window_data.T
                
                windows.append(window_data)
                
                # 将cluster_id映射到0, 1, 2, ...的标签
                cluster_label = cluster_ids.index(cluster_id)
                labels.append(cluster_label)
                
            except Exception as e:
                print(f"提取时间点 {spike_time} 时出错: {e}")
                continue
    
    if len(windows) == 0:
        print("警告: 没有成功提取任何时间窗数据")
        return np.array([]), np.array([])
    
    windows = np.stack(windows)
    labels = np.array(labels)
    
    print(f"成功提取 {len(windows)} 个时间窗，形状: {windows.shape}")
    print(f"标签分布: {np.bincount(labels)}")
    
    return windows, labels

print("Spike Classification模型类和数据准备函数已定义")


Spike Classification模型类和数据准备函数已定义


In [10]:
# 4. Spike Classification训练循环
if len(multi_cluster_groups) > 0:
    print("=== 开始Spike Classification训练 ===")
    
    # 创建classification结果保存目录
    classification_result_dir = '/media/ubuntu/sda/duan/script/spike_sorting/classification_results'
    os.makedirs(classification_result_dir, exist_ok=True)
    
    # 训练参数
    hidden_size1 = 256
    hidden_size2 = 64
    device = 'cuda'
    num_epochs = 100
    batch_size = 512
    window_size = 91
    
    # 存储所有classification结果
    classification_results = {}
    
    # 处理每个多cluster通道组合
    for idx, (channels_tuple, cluster_ids) in enumerate(multi_cluster_groups.items()):
        channel_group_id = str(list(channels_tuple))
        print(f"\n{'='*80}")
        print(f"处理第 {idx+1}/{len(multi_cluster_groups)} 个多cluster通道组合: {channel_group_id}")
        print(f"对应的clusters: {cluster_ids}")
        print(f"{'='*80}")
        
        try:
            # 创建该通道组合的结果保存目录
            result_dir = os.path.join(classification_result_dir, f"channels_{channel_group_id.replace(' ', '').replace('[', '').replace(']', '')}")
            os.makedirs(result_dir, exist_ok=True)
            
            # 获取该通道组合对应的通道ID列表
            channel_ids = list(channels_tuple)
            
            # 检查这些channel_ids是否在recording_f中存在
            valid_channels = []
            for ch_id in channel_ids:
                if ch_id < len(recording_f.channel_ids):
                    valid_channels.append(recording_f.channel_ids[ch_id])
                else:
                    print(f"警告: 通道ID {ch_id} 超出范围，跳过")
                    break
            
            if len(valid_channels) == 0:
                print(f"错误: 通道组合 {channel_ids} 中没有可用通道，跳过")
                continue
                
            print(f"使用通道: {valid_channels}")
            
            # 获取对应clusters的spike数据
            spike_inf_temp = spike_inf[spike_inf['cluster_id'].isin(cluster_ids)]
            print(f"对应clusters的spike数量: {len(spike_inf_temp)}")
            
            if len(spike_inf_temp) == 0:
                print("警告: 没有找到对应的spike数据，跳过此通道组合")
                continue
            
            # 检查每个cluster的spike数量
            cluster_counts = spike_inf_temp['cluster_id'].value_counts().sort_index()
            print(f"各cluster的spike数量: {dict(cluster_counts)}")
            
            # 检查是否有足够的样本进行训练
            min_samples = cluster_counts.min()
            if min_samples < 100:
                print(f"警告: 最少cluster只有 {min_samples} 个样本，可能影响训练效果")
            
            # 准备classification数据
            windows, labels = prepare_classification_data_from_spike_inf(
                recording_f, spike_inf_temp, cluster_ids, valid_channels, window_size
            )
            
            if len(windows) == 0:
                print("错误: 没有成功提取任何时间窗数据，跳过此通道组合")
                continue
            
            # 检查数据平衡性
            unique_labels, counts = np.unique(labels, return_counts=True)
            print(f"标签分布: {dict(zip(unique_labels, counts))}")
            
            # 如果数据不平衡，进行平衡采样
            min_count = min(counts)
            if min_count < max(counts) * 0.5:  # 如果最少类别的样本数少于最多类别的50%
                print(f"检测到数据不平衡，进行平衡采样...")
                balanced_indices = []
                for label in unique_labels:
                    label_indices = np.where(labels == label)[0]
                    if len(label_indices) > min_count:
                        # 随机采样min_count个样本
                        sampled_indices = np.random.choice(label_indices, min_count, replace=False)
                    else:
                        sampled_indices = label_indices
                    balanced_indices.extend(sampled_indices)
                
                balanced_indices = np.array(balanced_indices)
                np.random.shuffle(balanced_indices)
                
                windows = windows[balanced_indices]
                labels = labels[balanced_indices]
                
                print(f"平衡采样后数据量: {len(windows)}")
                print(f"平衡后标签分布: {np.bincount(labels)}")
            
            # 创建数据集
            dataset = SpikeDataset(windows, labels)
            
            train_size = int(0.8 * len(dataset))
            test_size = len(dataset) - train_size
            train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
            
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
            
            # 创建模型
            input_size = windows.shape[1] * windows.shape[2]
            num_classes = len(cluster_ids)
            model = Spike_Classification_MLP(input_size, hidden_size1, hidden_size2, 
                                            num_classes, n_channels=windows.shape[1], time_window=windows.shape[2])
            model = model.to(device)
            
            optimizer = optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.CrossEntropyLoss()
            
            # 训练模型
            print(f"\n开始训练classification模型...")
            training_history = []
            best_accuracy = 0
            best_model_state = None
            patience = 10  # 早停耐心值
            patience_counter = 0  # 早停计数器
            
            for epoch in range(num_epochs):
                # 训练阶段
                model.train()
                train_loss = 0
                train_correct = 0
                train_total = 0
                
                for batch_data, batch_labels in train_loader:
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)
                    
                    outputs = model(batch_data)
                    loss = criterion(outputs, batch_labels)
                    
                    _, predicted = torch.max(outputs.data, 1)
                    train_total += batch_labels.size(0)
                    train_correct += (predicted == batch_labels).sum().item()
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                    train_loss += loss.item()
                
                # 验证阶段
                model.eval()
                test_correct = 0
                test_total = 0
                all_predictions = []
                all_labels = []
                
                with torch.no_grad():
                    for batch_data, batch_labels in test_loader:
                        batch_data = batch_data.to(device)
                        batch_labels = batch_labels.to(device)
                        
                        outputs = model(batch_data)
                        _, predicted = torch.max(outputs.data, 1)
                        test_total += batch_labels.size(0)
                        test_correct += (predicted == batch_labels).sum().item()
                        
                        all_predictions.extend(predicted.cpu().numpy())
                        all_labels.extend(batch_labels.cpu().numpy())
                
                accuracy = test_correct / test_total if test_total > 0 else 0
                
                # 保存最佳模型和早停机制
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_model_state = model.state_dict().copy()
                    patience_counter = 0  # 重置计数器
                    print(f"Epoch {epoch}: 新的最佳准确率={accuracy:.4f}, 重置早停计数器")
                else:
                    patience_counter += 1
                    print(f"Epoch {epoch}: 准确率={accuracy:.4f} (最佳: {best_accuracy:.4f}), 早停计数器: {patience_counter}/{patience}")
                
                training_history.append({
                    'epoch': epoch,
                    'train_loss': train_loss / len(train_loader),
                    'train_accuracy': train_correct / train_total if train_total > 0 else 0,
                    'test_accuracy': accuracy,
                    'patience_counter': patience_counter
                })
                
                if epoch % 10 == 0 or epoch == num_epochs - 1:
                    print(f"Epoch {epoch}: Train Loss={train_loss/len(train_loader):.4f}, Train Acc={train_correct/train_total:.4f}, Test Acc={accuracy:.4f}")
                
                # 早停检查
                if patience_counter >= patience:
                    print(f"\n早停触发！连续 {patience} 个epoch没有提升，在第 {epoch+1} 个epoch停止训练")
                    print(f"最佳准确率: {best_accuracy:.4f}")
                    break
            
            # 保存最佳模型
            if best_model_state is not None:
                model.load_state_dict(best_model_state)
                torch.save(model.state_dict(), os.path.join(result_dir, 'best_classification_model.pth'))
            
            # 计算实际训练的epoch数
            actual_epochs = len(training_history)
            early_stopped = actual_epochs < num_epochs
            
            # 计算详细的分类指标
            from sklearn.metrics import classification_report, confusion_matrix
            
            # 使用最佳模型进行最终评估
            model.eval()
            final_predictions = []
            final_labels = []
            
            with torch.no_grad():
                for batch_data, batch_labels in test_loader:
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)
                    
                    outputs = model(batch_data)
                    _, predicted = torch.max(outputs.data, 1)
                    
                    final_predictions.extend(predicted.cpu().numpy())
                    final_labels.extend(batch_labels.cpu().numpy())
            
            # 生成分类报告
            cluster_names = [f"Cluster_{cid}" for cid in cluster_ids]
            classification_rep = classification_report(final_labels, final_predictions, 
                                                     target_names=cluster_names, output_dict=True)
            confusion_mat = confusion_matrix(final_labels, final_predictions)
            
            # 保存结果
            result_summary = {
                'channel_group_id': channel_group_id,
                'channels': valid_channels,
                'cluster_ids': cluster_ids,
                'cluster_names': cluster_names,
                'training_stats': {
                    'best_accuracy': float(best_accuracy),
                    'total_epochs': num_epochs,
                    'actual_epochs': actual_epochs,
                    'early_stopped': early_stopped,
                    'patience': patience
                },
                'data_stats': {
                    'total_samples': len(windows),
                    'train_samples': len(train_dataset),
                    'test_samples': len(test_dataset),
                    'window_shape': windows.shape,
                    'cluster_counts': dict(zip(cluster_ids, np.bincount(labels)))
                },
                'classification_report': classification_rep,
                'confusion_matrix': confusion_mat.tolist()
            }
            
            # 保存详细结果
            with open(os.path.join(result_dir, 'classification_result_summary.pkl'), 'wb') as f:
                pickle.dump(result_summary, f)
            
            with open(os.path.join(result_dir, 'classification_training_history.pkl'), 'wb') as f:
                pickle.dump(training_history, f)
            
            # 保存到总结果字典
            classification_results[channel_group_id] = result_summary
            
            print(f"通道组合 {channel_group_id} classification处理完成，最佳准确率: {best_accuracy:.4f}")
            print(f"分类报告:")
            print(classification_report(final_labels, final_predictions, target_names=cluster_names))
            
        except Exception as e:
            print(f"处理通道组合 {channel_group_id} 时出错: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n{'='*80}")
    print("所有多cluster通道组合classification处理完成！")
    print(f"{'='*80}")
    
    # 保存总结果
    with open(os.path.join(classification_result_dir, 'all_classification_results.pkl'), 'wb') as f:
        pickle.dump(classification_results, f)
    
    print(f"Classification结果已保存到: {classification_result_dir}")
else:
    print("没有找到包含多个cluster的通道组合，跳过classification训练")


=== 开始Spike Classification训练 ===

处理第 1/17 个多cluster通道组合: [199, 208, 210, 240, 241, 242]
对应的clusters: [2, 27, 40, 76, 78]
使用通道: ['B-071', 'B-080', 'B-082', 'B-112', 'B-113', 'B-114']
对应clusters的spike数量: 51987
各cluster的spike数量: {2: 10727, 27: 10642, 40: 11121, 76: 7714, 78: 11783}
开始从spike_inf提取数据，总共 51987 个spike...


提取时间窗: 100%|██████████| 51987/51987 [04:06<00:00, 210.49it/s]


成功提取 51987 个时间窗，形状: (51987, 6, 91)
标签分布: [10727 10642 11121  7714 11783]
标签分布: {0: 10727, 1: 10642, 2: 11121, 3: 7714, 4: 11783}

开始训练classification模型...
Epoch 0: 新的最佳准确率=0.6777, 重置早停计数器
Epoch 0: Train Loss=1.1018, Train Acc=0.5537, Test Acc=0.6777
Epoch 1: 新的最佳准确率=0.6910, 重置早停计数器
Epoch 2: 新的最佳准确率=0.6961, 重置早停计数器
Epoch 3: 新的最佳准确率=0.6989, 重置早停计数器
Epoch 4: 准确率=0.6983 (最佳: 0.6989), 早停计数器: 1/10
Epoch 5: 准确率=0.6955 (最佳: 0.6989), 早停计数器: 2/10
Epoch 6: 准确率=0.6962 (最佳: 0.6989), 早停计数器: 3/10
Epoch 7: 新的最佳准确率=0.7088, 重置早停计数器
Epoch 8: 准确率=0.7056 (最佳: 0.7088), 早停计数器: 1/10
Epoch 9: 准确率=0.6997 (最佳: 0.7088), 早停计数器: 2/10
Epoch 10: 准确率=0.7009 (最佳: 0.7088), 早停计数器: 3/10
Epoch 10: Train Loss=0.7083, Train Acc=0.6940, Test Acc=0.7009
Epoch 11: 准确率=0.7053 (最佳: 0.7088), 早停计数器: 4/10
Epoch 12: 准确率=0.7063 (最佳: 0.7088), 早停计数器: 5/10
Epoch 13: 准确率=0.7030 (最佳: 0.7088), 早停计数器: 6/10
Epoch 14: 准确率=0.7029 (最佳: 0.7088), 早停计数器: 7/10
Epoch 15: 准确率=0.7011 (最佳: 0.7088), 早停计数器: 8/10
Epoch 16: 准确率=0.7071 (最佳: 0.7088), 早停计数器: 9/

提取时间窗: 100%|██████████| 578865/578865 [41:49<00:00, 230.70it/s]


成功提取 578865 个时间窗，形状: (578865, 6, 91)
标签分布: [575209   3656]
标签分布: {0: 575209, 1: 3656}
检测到数据不平衡，进行平衡采样...
平衡采样后数据量: 7312
平衡后标签分布: [3656 3656]

开始训练classification模型...
Epoch 0: 新的最佳准确率=0.9686, 重置早停计数器
Epoch 0: Train Loss=0.4688, Train Acc=0.8024, Test Acc=0.9686
Epoch 1: 新的最佳准确率=0.9720, 重置早停计数器
Epoch 2: 新的最佳准确率=0.9802, 重置早停计数器
Epoch 3: 新的最佳准确率=0.9884, 重置早停计数器
Epoch 4: 准确率=0.9870 (最佳: 0.9884), 早停计数器: 1/10
Epoch 5: 准确率=0.9877 (最佳: 0.9884), 早停计数器: 2/10
Epoch 6: 准确率=0.9877 (最佳: 0.9884), 早停计数器: 3/10
Epoch 7: 准确率=0.9870 (最佳: 0.9884), 早停计数器: 4/10
Epoch 8: 准确率=0.9856 (最佳: 0.9884), 早停计数器: 5/10
Epoch 9: 准确率=0.9870 (最佳: 0.9884), 早停计数器: 6/10
Epoch 10: 新的最佳准确率=0.9897, 重置早停计数器
Epoch 10: Train Loss=0.0125, Train Acc=0.9964, Test Acc=0.9897
Epoch 11: 准确率=0.9870 (最佳: 0.9897), 早停计数器: 1/10
Epoch 12: 准确率=0.9884 (最佳: 0.9897), 早停计数器: 2/10
Epoch 13: 准确率=0.9877 (最佳: 0.9897), 早停计数器: 3/10
Epoch 14: 准确率=0.9870 (最佳: 0.9897), 早停计数器: 4/10
Epoch 15: 准确率=0.9870 (最佳: 0.9897), 早停计数器: 5/10
Epoch 16: 准确率=0.9870 (最佳: 0.9897

...

成功提取 25904 个时间窗，形状: (25904, 6, 91)
标签分布: [11457  9167  5280]
标签分布: {0: 11457, 1: 9167, 2: 5280}
检测到数据不平衡，进行平衡采样...
平衡采样后数据量: 15840
平衡后标签分布: [5280 5280 5280]

开始训练classification模型...
Epoch 0: 新的最佳准确率=0.8592, 重置早停计数器
Epoch 0: Train Loss=0.9184, Train Acc=0.5900, Test Acc=0.8592
Epoch 1: 新的最佳准确率=0.8987, 重置早停计数器
Epoch 2: 新的最佳准确率=0.9151, 重置早停计数器
Epoch 3: 准确率=0.9034 (最佳: 0.9151), 早停计数器: 1/10
Epoch 4: 新的最佳准确率=0.9274, 重置早停计数器
Epoch 5: 新的最佳准确率=0.9287, 重置早停计数器
Epoch 6: 准确率=0.9271 (最佳: 0.9287), 早停计数器: 1/10
Epoch 7: 准确率=0.9195 (最佳: 0.9287), 早停计数器: 2/10
Epoch 8: 准确率=0.9280 (最佳: 0.9287), 早停计数器: 3/10
Epoch 9: 准确率=0.9252 (最佳: 0.9287), 早停计数器: 4/10
Epoch 10: 准确率=0.9252 (最佳: 0.9287), 早停计数器: 5/10
Epoch 10: Train Loss=0.1605, Train Acc=0.9508, Test Acc=0.9252
Epoch 11: 准确率=0.9211 (最佳: 0.9287), 早停计数器: 6/10
Epoch 12: 准确率=0.9271 (最佳: 0.9287), 早停计数器: 7/10
Epoch 13: 准确率=0.9195 (最佳: 0.9287), 早停计数器: 8/10
Epoch 14: 准确率=0.9233 (最佳: 0.9287), 早停计数器: 9/10
Epoch 15: 准确率=0.9214 (最佳: 0.9287), 早停计数器: 10/10

早停触发！连续 10 个ep

提取时间窗: 100%|██████████| 366679/366679 [26:08<00:00, 233.84it/s]


成功提取 366679 个时间窗，形状: (366679, 6, 91)
标签分布: [  5674 361005]
标签分布: {0: 5674, 1: 361005}
检测到数据不平衡，进行平衡采样...
平衡采样后数据量: 11348
平衡后标签分布: [5674 5674]

开始训练classification模型...
Epoch 0: 新的最佳准确率=0.9877, 重置早停计数器
Epoch 0: Train Loss=0.3943, Train Acc=0.8592, Test Acc=0.9877
Epoch 1: 新的最佳准确率=0.9965, 重置早停计数器
Epoch 2: 准确率=0.9965 (最佳: 0.9965), 早停计数器: 1/10
Epoch 3: 新的最佳准确率=0.9978, 重置早停计数器
Epoch 4: 准确率=0.9978 (最佳: 0.9978), 早停计数器: 1/10
Epoch 5: 新的最佳准确率=0.9982, 重置早停计数器
Epoch 6: 准确率=0.9974 (最佳: 0.9982), 早停计数器: 1/10
Epoch 7: 准确率=0.9978 (最佳: 0.9982), 早停计数器: 2/10
Epoch 8: 准确率=0.9974 (最佳: 0.9982), 早停计数器: 3/10
Epoch 9: 准确率=0.9974 (最佳: 0.9982), 早停计数器: 4/10
Epoch 10: 新的最佳准确率=0.9987, 重置早停计数器
Epoch 10: Train Loss=0.0029, Train Acc=0.9989, Test Acc=0.9987
Epoch 11: 准确率=0.9978 (最佳: 0.9987), 早停计数器: 1/10
Epoch 12: 准确率=0.9982 (最佳: 0.9987), 早停计数器: 2/10
Epoch 13: 准确率=0.9987 (最佳: 0.9987), 早停计数器: 3/10
Epoch 14: 准确率=0.9978 (最佳: 0.9987), 早停计数器: 4/10
Epoch 15: 准确率=0.9974 (最佳: 0.9987), 早停计数器: 5/10
Epoch 16: 准确率=0.9978 (最佳: 0.998

提取时间窗: 100%|██████████| 428296/428296 [30:33<00:00, 233.63it/s]


成功提取 428296 个时间窗，形状: (428296, 6, 91)
标签分布: [424755   3541]
标签分布: {0: 424755, 1: 3541}
检测到数据不平衡，进行平衡采样...
平衡采样后数据量: 7082
平衡后标签分布: [3541 3541]

开始训练classification模型...
Epoch 0: 新的最佳准确率=0.9668, 重置早停计数器
Epoch 0: Train Loss=0.3360, Train Acc=0.8378, Test Acc=0.9668
Epoch 1: 新的最佳准确率=0.9901, 重置早停计数器
Epoch 2: 新的最佳准确率=0.9915, 重置早停计数器
Epoch 3: 准确率=0.9866 (最佳: 0.9915), 早停计数器: 1/10
Epoch 4: 准确率=0.9915 (最佳: 0.9915), 早停计数器: 2/10
Epoch 5: 准确率=0.9894 (最佳: 0.9915), 早停计数器: 3/10
Epoch 6: 新的最佳准确率=0.9922, 重置早停计数器
Epoch 7: 新的最佳准确率=0.9936, 重置早停计数器
Epoch 8: 准确率=0.9929 (最佳: 0.9936), 早停计数器: 1/10
Epoch 9: 准确率=0.9936 (最佳: 0.9936), 早停计数器: 2/10
Epoch 10: 新的最佳准确率=0.9944, 重置早停计数器
Epoch 10: Train Loss=0.0052, Train Acc=0.9986, Test Acc=0.9944
Epoch 11: 准确率=0.9944 (最佳: 0.9944), 早停计数器: 1/10
Epoch 12: 准确率=0.9944 (最佳: 0.9944), 早停计数器: 2/10
Epoch 13: 准确率=0.9915 (最佳: 0.9944), 早停计数器: 3/10
Epoch 14: 准确率=0.9944 (最佳: 0.9944), 早停计数器: 4/10
Epoch 15: 准确率=0.9944 (最佳: 0.9944), 早停计数器: 5/10
Epoch 16: 准确率=0.9929 (最佳: 0.9944), 早停计数器: 6/1

In [11]:
# 5. 分类结果评估和可视化 - 可调整参数版本
if len(multi_cluster_groups) > 0:
    print("=== 生成Classification结果评估和可视化 ===")
    
    # 加载classification结果
    try:
        with open('/media/ubuntu/sda/duan/script/spike_sorting/classification_results/all_classification_results.pkl', 'rb') as f:
            classification_results = pickle.load(f)
        
        print(f"成功加载 {len(classification_results)} 个classification结果")
        
        # 提取分类指标数据
        classification_metrics = []
        
        for channel_group_id, result_summary in classification_results.items():
            metrics_data = {
                'channel_group_id': channel_group_id,
                'best_accuracy': result_summary['training_stats']['best_accuracy'],
                'num_clusters': len(result_summary['cluster_ids']),
                'total_samples': result_summary['data_stats']['total_samples'],
                'actual_epochs': result_summary['training_stats']['actual_epochs'],
                'early_stopped': result_summary['training_stats']['early_stopped']
            }
            
            # 提取每个cluster的精确率、召回率和F1分数
            classification_rep = result_summary['classification_report']
            for cluster_name in result_summary['cluster_names']:
                if cluster_name in classification_rep:
                    cluster_metrics = classification_rep[cluster_name]
                    metrics_data[f'{cluster_name}_precision'] = cluster_metrics['precision']
                    metrics_data[f'{cluster_name}_recall'] = cluster_metrics['recall']
                    metrics_data[f'{cluster_name}_f1'] = cluster_metrics['f1-score']
            
            classification_metrics.append(metrics_data)
        
        # 转换为DataFrame
        classification_df = pd.DataFrame(classification_metrics)
        
        print("Classification指标提取完成！")
        print(f"Classification指标形状: {classification_df.shape}")
        
        # 生成可视化图表 - 可调整参数版本
        # 图像参数设置
        plot_params = {
            'figsize_width': 10,      # 图像宽度
            'figsize_height': 6,      # 图像高度
            'title_fontsize': 16,     # 标题字体大小
            'label_fontsize': 14,     # 标签字体大小
            'tick_fontsize': 12,      # 刻度字体大小
            'box_alpha': 0.8,         # 箱线图透明度
            'scatter_alpha': 0.7,     # 散点图透明度
            'scatter_size': 80,       # 散点大小
            'line_width': 2,          # 线条宽度
            'grid_alpha': 0.3,        # 网格透明度
            'colors': ['#7DAEE0', '#B395BD', '#EA8379', '#95C4A3'],  # 颜色列表
            'colormap': 'viridis',    # 颜色映射
            'dpi': 300,               # 图像分辨率
            'bbox_inches': 'tight'    # 图像边距
        }
        
        with PdfPages('/media/ubuntu/sda/duan/figure/spike_classification_eval.pdf') as pdf:
            # 图1: 整体准确率分布 - 横向布局
            fig, ax = plt.subplots(1, 1, figsize=(plot_params['figsize_width'], plot_params['figsize_height']))
            
            accuracy_data = classification_df['best_accuracy'] * 100  # 转换为百分比
            
            # 创建横向箱线图
            bp = ax.boxplot([accuracy_data], labels=['Classification Accuracy'], 
                          patch_artist=True, vert=False)  # vert=False 使图像横向
            bp['boxes'][0].set_facecolor(plot_params['colors'][0])
            bp['boxes'][0].set_alpha(plot_params['box_alpha'])
            
            # 设置中位数线颜色
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(plot_params['line_width'])
            
            ax.set_title('Spike Classification Accuracy Distribution', 
                        fontsize=plot_params['title_fontsize'], fontweight='bold')
            ax.set_xlabel('Accuracy (%)', fontsize=plot_params['label_fontsize'])
            ax.set_xlim(0, 105)
            ax.grid(True, alpha=plot_params['grid_alpha'])
            ax.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            plt.tight_layout()
            pdf.savefig(dpi=plot_params['dpi'], bbox_inches=plot_params['bbox_inches'])
            plt.close()
            
            # 图2: 准确率vs聚类数量 - 横向布局
            fig, ax = plt.subplots(1, 1, figsize=(plot_params['figsize_width'], plot_params['figsize_height']))
            
            scatter = ax.scatter(classification_df['num_clusters'], 
                               classification_df['best_accuracy'] * 100,
                               c=classification_df['total_samples'], 
                               cmap=plot_params['colormap'], 
                               alpha=plot_params['scatter_alpha'],
                               s=plot_params['scatter_size'],
                               edgecolors='black',
                               linewidth=0.5)
            
            ax.set_xlabel('Number of Clusters', fontsize=plot_params['label_fontsize'])
            ax.set_ylabel('Classification Accuracy (%)', fontsize=plot_params['label_fontsize'])
            ax.set_title('Classification Accuracy vs Number of Clusters', 
                        fontsize=plot_params['title_fontsize'], fontweight='bold')
            ax.grid(True, alpha=plot_params['grid_alpha'])
            ax.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            # 添加颜色条
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label('Total Samples', fontsize=plot_params['label_fontsize'])
            cbar.ax.tick_params(labelsize=plot_params['tick_fontsize'])
            
            plt.tight_layout()
            pdf.savefig(dpi=plot_params['dpi'], bbox_inches=plot_params['bbox_inches'])
            plt.close()
            
            # 图3: 训练epochs分布 - 横向布局
            fig, ax = plt.subplots(1, 1, figsize=(plot_params['figsize_width'], plot_params['figsize_height']))
            
            epochs_data = classification_df['actual_epochs']
            
            bp = ax.boxplot([epochs_data], labels=['Training Epochs'], 
                          patch_artist=True, vert=False)  # vert=False 使图像横向
            bp['boxes'][0].set_facecolor(plot_params['colors'][1])
            bp['boxes'][0].set_alpha(plot_params['box_alpha'])
            
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(plot_params['line_width'])
            
            ax.set_title('Training Epochs Distribution', 
                        fontsize=plot_params['title_fontsize'], fontweight='bold')
            ax.set_xlabel('Number of Epochs', fontsize=plot_params['label_fontsize'])
            ax.grid(True, alpha=plot_params['grid_alpha'])
            ax.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            plt.tight_layout()
            pdf.savefig(dpi=plot_params['dpi'], bbox_inches=plot_params['bbox_inches'])
            plt.close()
            
            # 图4: 混淆矩阵热图 - 横向布局
            if len(classification_results) > 0:
                first_result = list(classification_results.values())[0]
                confusion_mat = np.array(first_result['confusion_matrix'])
                cluster_names = first_result['cluster_names']
                
                fig, ax = plt.subplots(1, 1, figsize=(plot_params['figsize_width'], plot_params['figsize_height']))
                
                im = ax.imshow(confusion_mat, cmap='Blues', aspect='auto')
                
                # 设置标签
                ax.set_xticks(range(len(cluster_names)))
                ax.set_yticks(range(len(cluster_names)))
                ax.set_xticklabels(cluster_names, rotation=45, fontsize=plot_params['tick_fontsize'])
                ax.set_yticklabels(cluster_names, fontsize=plot_params['tick_fontsize'])
                
                # 添加数值标签
                for i in range(len(cluster_names)):
                    for j in range(len(cluster_names)):
                        text = ax.text(j, i, confusion_mat[i, j],
                                     ha="center", va="center", color="black", 
                                     fontweight='bold', fontsize=plot_params['tick_fontsize'])
                
                ax.set_xlabel('Predicted Cluster', fontsize=plot_params['label_fontsize'])
                ax.set_ylabel('True Cluster', fontsize=plot_params['label_fontsize'])
                ax.set_title(f'Confusion Matrix - {first_result["channel_group_id"]}', 
                           fontsize=plot_params['title_fontsize'], fontweight='bold')
                
                # 添加颜色条
                cbar = plt.colorbar(im, ax=ax)
                cbar.set_label('Count', fontsize=plot_params['label_fontsize'])
                cbar.ax.tick_params(labelsize=plot_params['tick_fontsize'])
                
                plt.tight_layout()
                pdf.savefig(dpi=plot_params['dpi'], bbox_inches=plot_params['bbox_inches'])
                plt.close()
            
            # 图5: 多子图组合 - 横向布局
            fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(plot_params['figsize_width']*1.5, plot_params['figsize_height']*1.2))
            
            # 子图1: 准确率分布
            bp1 = ax1.boxplot([accuracy_data], labels=['Accuracy'], patch_artist=True, vert=False)
            bp1['boxes'][0].set_facecolor(plot_params['colors'][0])
            bp1['boxes'][0].set_alpha(plot_params['box_alpha'])
            ax1.set_xlabel('Accuracy (%)', fontsize=plot_params['label_fontsize'])
            ax1.set_title('Accuracy Distribution', fontsize=plot_params['title_fontsize'])
            ax1.grid(True, alpha=plot_params['grid_alpha'])
            ax1.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            # 子图2: Epochs分布
            bp2 = ax2.boxplot([epochs_data], labels=['Epochs'], patch_artist=True, vert=False)
            bp2['boxes'][0].set_facecolor(plot_params['colors'][1])
            bp2['boxes'][0].set_alpha(plot_params['box_alpha'])
            ax2.set_xlabel('Number of Epochs', fontsize=plot_params['label_fontsize'])
            ax2.set_title('Training Epochs', fontsize=plot_params['title_fontsize'])
            ax2.grid(True, alpha=plot_params['grid_alpha'])
            ax2.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            # 子图3: 准确率vs聚类数量
            scatter3 = ax3.scatter(classification_df['num_clusters'], 
                                 classification_df['best_accuracy'] * 100,
                                 c=classification_df['total_samples'], 
                                 cmap=plot_params['colormap'], 
                                 alpha=plot_params['scatter_alpha'],
                                 s=plot_params['scatter_size'])
            ax3.set_xlabel('Number of Clusters', fontsize=plot_params['label_fontsize'])
            ax3.set_ylabel('Accuracy (%)', fontsize=plot_params['label_fontsize'])
            ax3.set_title('Accuracy vs Clusters', fontsize=plot_params['title_fontsize'])
            ax3.grid(True, alpha=plot_params['grid_alpha'])
            ax3.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            # 子图4: 样本数量分布
            sample_counts = classification_df['total_samples']
            bp4 = ax4.boxplot([sample_counts], labels=['Samples'], patch_artist=True, vert=False)
            bp4['boxes'][0].set_facecolor(plot_params['colors'][2])
            bp4['boxes'][0].set_alpha(plot_params['box_alpha'])
            ax4.set_xlabel('Number of Samples', fontsize=plot_params['label_fontsize'])
            ax4.set_title('Sample Count Distribution', fontsize=plot_params['title_fontsize'])
            ax4.grid(True, alpha=plot_params['grid_alpha'])
            ax4.tick_params(axis='both', which='major', labelsize=plot_params['tick_fontsize'])
            
            plt.suptitle('Spike Classification Analysis Summary', 
                        fontsize=plot_params['title_fontsize']+2, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(dpi=plot_params['dpi'], bbox_inches=plot_params['bbox_inches'])
            plt.close()
        
        print("Classification评估图表已保存到: /media/ubuntu/sda/duan/figure/spike_classification_eval.pdf")
        
        # 打印总体统计信息
        print(f"\n=== Classification总体统计 ===")
        print(f"处理的通道组合数量: {len(classification_df)}")
        print(f"平均分类准确率: {classification_df['best_accuracy'].mean()*100:.2f}%")
        print(f"最高分类准确率: {classification_df['best_accuracy'].max()*100:.2f}%")
        print(f"最低分类准确率: {classification_df['best_accuracy'].min()*100:.2f}%")
        print(f"平均聚类数量: {classification_df['num_clusters'].mean():.1f}")
        print(f"平均训练样本数: {classification_df['total_samples'].mean():.0f}")
        print(f"平均训练epochs: {classification_df['actual_epochs'].mean():.1f}")
        print(f"早停比例: {classification_df['early_stopped'].sum()}/{len(classification_df)} ({classification_df['early_stopped'].mean()*100:.1f}%)")
        
    except FileNotFoundError:
        print("错误: 找不到classification结果文件，请先运行classification训练")
    except Exception as e:
        print(f"生成评估图表时出错: {e}")
        import traceback
        traceback.print_exc()
else:
    print("没有classification结果可供评估")


=== 生成Classification结果评估和可视化 ===
成功加载 17 个classification结果
Classification指标提取完成！
Classification指标形状: (17, 135)
Classification评估图表已保存到: /media/ubuntu/sda/duan/figure/spike_classification_eval.pdf

=== Classification总体统计 ===
处理的通道组合数量: 17
平均分类准确率: 95.74%
最高分类准确率: 100.00%
最低分类准确率: 70.88%
平均聚类数量: 2.5
平均训练样本数: 24781
平均训练epochs: 22.1
早停比例: 17/17 (100.0%)
